# Full-Surah → Verses Restructure — ALL Reciters (Colab T4)

Restructures every flat reciter in `zaibihassan/Quranic-Recitation-Data` into the Alnufais two-folder layout (`{Reciter}/Full Surah/*` + `{Reciter}/Verses/*`), **one reciter per run**. All compute happens here on the Colab T4 GPU — nothing runs locally.

**Per run:** pass 1 aligns 114 full surahs (CUDA, batch 8) for cut points → zero-bleed ayah clips → pass 2 re-aligns each verse clip against an ayah-slice reference (verse-relative timings out of Viterbi, never offset subtraction).

**Time:** ~2–3h per reciter on T4. Resume-safe: re-running the last cell finishes the remainder after a timeout.

**Before starting:** Runtime → *Change runtime type* → **T4 GPU**, and add a Colab secret named `HF_TOKEN` (fine-grained token with **write** access to the dataset).

In [ ]:
# 0. Install package (GPU build) + helpers
!pip install -q git+https://github.com/HsnSaboor/quran-forced-align.git 2>&1 | tail -1
!pip install -q huggingface_hub
!apt-get install -y -qq ffmpeg > /dev/null 2>&1
print("✓ installed")

In [ ]:
# 1. Mount Drive (resume cache across sessions/timeouts) + pick the reciter
from google.colab import drive
drive.mount('/content/drive')

# ▶▶▶ EDIT THIS LINE — one exact HF folder name per run, e.g.:
#   "Mishari Rashid Al Afasy (Murattal)"
#   "Abdul Rahman Al Sudais (Murattal)"
#   "Saad Al Ghamdi (Murattal)"
RECITER = "Mishari Rashid Al Afasy (Murattal)"

DRIVE_ROOT = "/content/drive/MyDrive/quran_verses"
OUT_DIR    = "/content/verses_out"       # local NVMe (fast); mirrored to Drive
AUDIO_DIR  = "/content/verses_audio"

import os
os.makedirs(f"{DRIVE_ROOT}/{RECITER}", exist_ok=True)
print(f"✓ reciter: {RECITER}")

In [ ]:
# 2. GPU check (fail fast if not on T4)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 3. Run the restructure (pass 1 + cut + pass 2 + upload). Re-run safe.
#    Skips: pass-1 jsons, verse clips, and per-surah index files already present.
!python -m scripts.colab_verses_all --reciter "$RECITER" \
    --out-dir /content/verses_out --audio-dir /content/verses_audio \
    --drive-dir /content/drive/MyDrive/quran_verses \
    --batch-size 8 --upload

In [ ]:
# 4. Verify the run (counts + spot-check one verse index)
import glob, json
clips = glob.glob(f"/content/verses_out/{RECITER}/Verses/*/*.opus")
idx   = sorted(glob.glob(f"/content/verses_out/{RECITER}/Verses/*/*.json"))
print(f"verse clips: {len(clips)} (expect 6236 for a complete reciter)")
print(f"verse index jsons: {len(idx)} (expect 114)")
if idx:
    d = json.load(open(idx[0]))
    print("sample:", idx[0].split("Verses/")[-1])
    print("  verses:", len(d["verses"]), "words:", d["metadata"]["totalWords"])
    print("  verse0:", json.dumps(d["verses"][0], ensure_ascii=False)[:300])